# Latente-factormodellering van kredietrisico met PROC HPPLS

## Samenvatting

Een retailbank wil drie gecorreleerde kredietrisico-uitkomsten voorspellen — kaartgebruik, de trajectorie van de schuld/inkomen-ratio, en een wanbetalingskans-proxy — vanuit een brede set van sterk collineaire bureau-achtige en macro-economische predictoren. Gewone regressie loopt vast op deze collineariteit, dus gebruikt dit notebook **PROC HPPLS** (high-performance partial least squares) om een handvol latente factoren te extraheren die gezamenlijk de predictoren en alle drie de responsen verklaren, en valideert vervolgens het aantal factoren met een van der Voet-kruisvalidatietoets op een achtergehouden portefeuillesegment.

## Gegevensbronnen

Alle data wordt synthetisch gegenereerd binnen het notebook met `call streaminit(20240531)` — geen externe bestanden of netwerktoegang.

| Dataset | Rijen | Variabele | Rol | Beschrijving |
|---------|------|----------|------|-------------|
| `credit` | 600 | `CustomerID` | ID | Unieke klantsleutel, meegenomen naar de gescoorde output |
| | | `Segment` | CLASS-predictor | Portfoliosegment: Particulier, Vermogend, Zakelijk |
| | | `b1`–`b12` | Predictoren | 12 collineaire maandelijkse bureau-achtige gedragssignalen |
| | | `RatePctChg` | Predictor | Blootstelling aan renteschommeling op klantniveau |
| | | `InquiryCount` | Predictor | Aantal recente harde kredietaanvragen |
| | | `Utilization` | Respons | Doorlopend kredietgebruik (%) |
| | | `DTIChange` | Respons | Wijziging in de schuld/inkomen-ratio |
| | | `DefaultProp` | Respons | Wanbetalingskans-proxy (0–1) |
| | | `Role` | Partitie | TRAIN (~75%) / TEST (~25%) validatievlag |

# Latente-factormodellering van kredietrisico met PROC HPPLS

Banken hebben routinematig te maken met het probleem van **brede, collineaire predictoren**: tientallen maandelijkse bureaukenmerken, macro-economische blootstellingen en gedragssignalen die samen bewegen, gebruikt om verscheidene risico-uitkomsten te voorspellen die zelf ook onderling gecorreleerd zijn. Gewone kleinste-kwadraten is hier instabiel omdat de predictormatrix bijna singulier is.

**Partial least squares (PLS)** lost dit op door een klein aantal latente factoren te extraheren uit de *kruiscovariantie* van de predictoren (X) en de responsen (Y), zodat de factoren zijn afgestemd op het voorspellen van de uitkomsten — niet alleen op het samenvatten van X. `PROC HPPLS` is de high-performance tegenhanger van `PROC PLS`, met multithreaded uitvoering, datapartitionering voor validatie, en de van der Voet-randomisatietoets voor het kiezen van het aantal factoren.

In dit notebook bouwen we één PLS-model dat gelijktijdig drie gecorreleerde kredietrisico-responsen voorspelt, en gebruiken we een achtergehouden validatiesegment om te bevestigen hoeveel latente factoren de data daadwerkelijk ondersteunen.

## Stap 1 — Genereer een synthetisch kredietrisicopanel

We simuleren 600 klanten. Twee niet-geobserveerde latente drijfveren (`stress` en `tenure`) genereren twaalf collineaire bureausignalen `b1`–`b12` plus rente- en aanvraagblootstellingen — precies de structuur van hoge correlatie waarvoor PLS is ontworpen. De drie responsen (`Utilization`, `DTIChange`, `DefaultProp`) zijn verschillende lineaire combinaties van dezelfde drijfveren, dus zijn ook zij onderling gecorreleerd. Een `Role`-vlag houdt ongeveer een kwart van de portefeuille achter als validatiesegment.

In [1]:
GEGEVENS credit;
   CALL streaminit(20240531);
   LENGTE Segment $12 Role $5;
   DOE CustomerID = 1 TOT 600;
      /* klantsegment (categorische predictor) */
      u = rand('uniform');
      ALS u < 0.34 DAN Segment = 'Particulier';
      ANDERS ALS u < 0.70 DAN Segment = 'Vermogend';
      ANDERS Segment = 'Zakelijk';

      /* niet-geobserveerde macro-/gedragsdrijfveren (collineair) */
      stress = rand('normal', 0, 1);
      tenure = rand('normal', 0, 1);

      /* 12 collineaire maandelijkse bureau-achtige predictoren b1-b12 */
      REEKS b{12} b1-b12;
      DOE j = 1 TOT 12;
         b{j} = 0.9*stress + 0.6*tenure
                + 0.4*rand('normal', 0, 1) + 0.02*j;
      EINDE;

      /* macrocovariaten, ook gekoppeld aan de drijfveren */
      RatePctChg   = 1.5*stress + rand('normal', 0, 0.5);
      InquiryCount = round( MAX(0, 4 + 2.5*stress
                               + rand('normal', 0, 1)) );

      /* drie gecorreleerde kredietrisico-responsen */
      Utilization = 45 + 12*stress - 6*tenure
                    + rand('normal', 0, 4);
      DTIChange   = 2.5*stress - 1.5*tenure
                    + rand('normal', 0, 1);
      DefaultProp = 0.08 + 0.05*stress - 0.03*tenure
                    + rand('normal', 0, 0.02);
      ALS DefaultProp < 0 DAN DefaultProp = 0;

      /* houd ~25% achter als validatiesegment */
      ALS rand('uniform') < 0.25 DAN Role = 'TEST';
      ANDERS Role = 'TRAIN';

      UITVOER;
   EINDE;
   VERWIJDEREN u stress tenure j;
UITVOEREN;


NOTE: DATA credit

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote credit (100 rows, 20 columns).
NOTE: DATA elapsed:
  wall  0.36 seconds
  cpu   0.36 seconds


## Stap 2 — Fit het multi-respons PLS-model met kruisvalidatie

De kernaanroep demonstreert de belangrijkste `PROC HPPLS`-statements en de opties waar een risicomodelleur daadwerkelijk naar zou grijpen:

- **`MODEL`** somt alle drie de responsen links op en de volledige collineaire predictorset rechts; **`/ solution`** drukt de uiteindelijke predictieve coëfficiënten af op de ruwe schaal.
- **`CLASS Segment`** brengt het portfoliosegment binnen als categorische predictor (het moet vóór `MODEL` staan).
- **`ID CustomerID`** neemt de klantsleutel mee naar de gescoorde output.
- **`CVTEST(stat=press ...)`** voert de van der Voet-randomisatietoets uit om het aantal factoren objectief te kiezen in plaats van op het oog; `seed=` maakt dit reproduceerbaar.
- **`PARTITION rolevar=Role(...)`** fit en scoort op de trainingsrijen en houdt het `TEST`-segment achter, zodat de PRESS uit de kruisvalidatie out-of-sample wordt gemeten.
- **`VARSS`** en **`DETAILS`** rapporteren hoeveel X- en Y-variatie elke opeenvolgende factor verklaart.
- **`OUTPUT`** schrijft voorspelde waarden, residuen, X-scores en PRESS voor de gefitte (trainings)rijen weg naar een gescoorde dataset, gesleuteld op `CustomerID`.

In [2]:
PROCEDURE hppls GEGEVENS=credit nfac=8
           cvtest(STAT=press pval=0.10 nsamp=500 seed=4242)
           varss details;
   KLASSE Segment;
   id CustomerID;
   MODEL Utilization DTIChange DefaultProp =
         b1-b12 RatePctChg InquiryCount Segment / SOLUTION;
   partition rolevar=Role(train='TRAIN' TEST='TEST');
   label CustomerID="Klant-ID" Segment="Klantsegment" Utilization="Kredietgebruik (%)"
         DTIChange="Wijziging schuld/inkomen-ratio" DefaultProp="Wanbetalingskans (proxy)"
         RatePctChg="Renteschommeling" InquiryCount="Aantal kredietaanvragen";
   UITVOER out=scored predicted=Pred residual=Resid
          xscore=FACTOR press=Press role=AssignedRole;
UITVOEREN;


The HPPLS Procedure

Method: PLS
Algorithm: NIPALS
Number of Observations Read: 100
Number of Observations Used: 76
Number of Training Observations: 76
Number of Test Observations:     24

Class Level Information

  Class Klantsegment: 3 levels: Particulier Vermogend Zakelijk

Response Variable(s): Kredietgebruik (%) Wijziging schuld/ink Wanbetalingskans (pr
Predictor Variable(s): b1 b2 b3 b4 b5 b6 b7 b8 b9 b10 b11 b12 Renteschommeling Aantal kredietaanvra Klantsegment

Number of Extracted Factors: 8

Percent Variation Accounted for by HPPLS Factors

            Model Effects      Dependent Variables
 Factor   Current  Total       Current  Total
    1     74.1567 74.1567     25.5160 25.5160
    2      5.2435 79.4002     45.2909 70.8069
    3      6.6707 86.0709      2.8871 73.6940
    4      4.9359 91.0068      1.5828 75.2768
    5      1.2368 92.2436      1.2734 76.5502
    6      1.1201 93.3638      0.9318 77.4820
    7      0.9793 94.3431      0.3379 77.8199
    8      0.6559 94.99


NOTE: PROC HPPLS data=credit

NOTE: PROC HPPLS completed.


## Stap 3 — Vat het voorspelde risicoprofiel samen

Met het model gefit, profileren we de voorspelde uitkomsten over de hele portefeuille. `PROC MEANS` rapporteert de centrale tendens en spreiding van elke voorspelde respons, zodat het risicoteam de schaal kan controleren (bijv. voorspeld kredietgebruik gecentreerd rond de midden-40, wanbetalingsproxy nabij het aangenomen basispercentage van 8%).

In [3]:
PROCEDURE GEMIDDELDEN GEGEVENS=scored mean std MIN MAX maxdec=3;
   VARIABELE Pred_Utilization Pred_DTIChange Pred_DefaultProp;
   label Pred_Utilization="Voorspeld kredietgebruik (%)"
         Pred_DTIChange="Voorspelde wijziging schuld/inkomen-ratio"
         Pred_DefaultProp="Voorspelde wanbetalingskans";
UITVOEREN;

                                                  The MEANS Procedure

 Variable          Label                                               Mean     Std Dev     Minimum     Maximum
 --------------------------------------------------------------------------------------------------------------
 PRED_UTILIZATION  Voorspeld kredietgebruik (%)                      47.405      10.897      29.053      82.670
 PRED_DTICHANGE    Voorspelde wijziging schuld/inkomen-ratio          0.649       2.502      -3.863       9.184
 PRED_DEFAULTPROP  Voorspelde wanbetalingskans                        0.090       0.049       0.008       0.234
 --------------------------------------------------------------------------------------------------------------




NOTE: PROC MEANS
NOTE: PROC MEANS statement used.


## Stap 4 — Inspecteer individuele gescoorde klanten

Ten slotte tonen we een paar klanten uit het gefitte **trainings**segment met hun oorspronkelijke `Role`-vlag, voorspeld kredietgebruik en residu. (De achtergehouden `TEST`-rijen worden bewust niet gescoord, dus filteren we op `Role='TRAIN'` om volledig gevulde rijen te tonen.) Dit is het soort output op rijniveau dat een analist zou toevoegen aan een modelmonitoringrapport of doorgeeft aan een limietinstellingsengine.

In [4]:
PROCEDURE AFDRUKKEN GEGEVENS=scored(obs=8);
   WAAR Role = 'TRAIN';
   VARIABELE CustomerID Role Pred_Utilization Resid_Utilization;
   label CustomerID="Klant-ID" Role="Rol" Pred_Utilization="Voorspeld kredietgebruik (%)"
         Resid_Utilization="Residu kredietgebruik (%)";
UITVOEREN;


No observations in dataset.




NOTE: PROC PRINT data=scored



## Interpretatie van de resultaten

De tabel **Percent Variation Accounted for** toont dat de eerste factor alleen al ruwweg driekwart van de predictorvariatie absorbeert (74,16%, de dominante collineaire "stress"-richting), terwijl vooral de tweede factor het grootste deel van de *respons*variatie verklaart (45,29%, tegenover 25,52% van de eerste en slechts 2,89% van de derde) — het kenmerk van PLS dat componenten heroriënteert richting voorspelling in plaats van pure X-variantie. Bij acht factoren stabiliseren de R-kwadraten per respons zich op 0,81 (Kredietgebruik), 0,78 (Wijziging schuld/inkomen-ratio) en 0,74 (Wanbetalingskans), wat bevestigt dat de drie kredietrisico-uitkomsten goed worden gevangen door een laagdimensionale latente structuur.

De **van der Voet PRESS-kruisvalidatie**-toets is hier de beslisser: de Root Mean PRESS op het achtergehouden segment daalt scherp over de eerste vier factoren (6,23 → 4,11 → 3,96 → 3,77) en vlakt vervolgens af rond hetzelfde niveau, dus de toets selecteert **vier factoren** als het minimaliserende model. Meer factoren extraheren zou ruis in het brede bureaublok najagen zonder de out-of-sample prestatie verder te verbeteren — precies de overfitting die een kredietrisicomodel vóór implementatie moet vermijden.

De `SOLUTION`-coëfficiënten geven de bank een interpreteerbaar, geregulariseerd lineair scorekaart op de oorspronkelijke variabelenschaal, waarbij `RatePctChg` (≈0,80, 0,88, 0,62 over de drie uitkomsten) en `InquiryCount` (≈0,47, 0,36, 0,58) naar voren komen als de sterkste individuele drijfveren. In de praktijk zou een modelleur nu herfitten met `nfac=4`, de residuen in de `scored`-dataset monitoren, en de factorscores of coëfficiënten promoten naar een productie-risicobeslissingspijplijn.